In [ ]:
!pip install shap xgboost scikit-learn pandas numpy matplotlib -q

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
import shap
from scipy.stats import spearmanr

In [ ]:
def load_german_credit():
    df = pd.read_csv('german.csv')
    df = df.sample(n=min(500, len(df)), random_state=42)
    return df

def load_uci_credit_card():
    df = pd.read_csv('default of credit card clients.csv')
    df = df.sample(n=500, random_state=42)
    return df

def load_give_me_credit():
    df = pd.read_csv('cs-training.csv')
    df = df.drop(columns=['Unnamed: 0'])
    df = df.sample(n=500, random_state=42)
    return df

In [ ]:
def preprocess(df, target_col):
    df = df.dropna()

    if df[target_col].dtype == 'object':
        le_target = LabelEncoder()
        df[target_col] = le_target.fit_transform(df[target_col])

    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])

    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

    return X_train_scaled, X_test_scaled, y_train, y_test, X_train, X_test

In [ ]:
def train_models(X_train, y_train, X_train_raw):
    models = {}

    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_train, y_train)
    models['LogisticRegression'] = lr

    rf = RandomForestClassifier(n_estimators=200, random_state=42)
    rf.fit(X_train_raw, y_train)
    models['RandomForest'] = rf

    xgb = XGBClassifier(random_state=42, eval_metric='logloss')
    xgb.fit(X_train_raw, y_train)
    models['XGBoost'] = xgb

    return models

In [ ]:
def compute_shap_values(model, model_name, X_background, X_explain):
    if model_name == 'LogisticRegression':
        explainer = shap.LinearExplainer(model, X_background)
    elif model_name == 'RandomForest':
        explainer = shap.TreeExplainer(model)
    elif model_name == 'XGBoost':
        explainer = shap.TreeExplainer(model)

    shap_values = explainer.shap_values(X_explain)

    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
        shap_values = shap_values[:, :, 1]

    return shap_values


def get_feature_ranking(shap_values, feature_names):
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    ranking = pd.Series(mean_abs_shap, index=feature_names).sort_values(ascending=False)
    return ranking

In [ ]:
def spearman_stability(ranking1, ranking2):
    common_features = ranking1.index.intersection(ranking2.index)
    r1 = ranking1[common_features]
    r2 = ranking2[common_features]
    corr, _ = spearmanr(r1, r2)
    return corr

def jaccard_overlap(ranking1, ranking2, k):
    top_k_1 = set(ranking1.head(k).index)
    top_k_2 = set(ranking2.head(k).index)
    intersection = len(top_k_1 & top_k_2)
    union = len(top_k_1 | top_k_2)
    return intersection / union

In [ ]:
datasets = {
    'German Credit': (load_german_credit(), 'target'),
    'UCI Credit Card': (load_uci_credit_card(), 'dpnm'),
    'GiveMeCredit': (load_give_me_credit(), 'SeriousDlqin2yrs'),
}

results = {}

for dataset_name, (df, target_col) in datasets.items():
    print(f"Processing {dataset_name}...")

    X_train_s, X_test_s, y_train, y_test, X_train_raw, X_test_raw = preprocess(df, target_col)

    models = train_models(X_train_s, y_train, X_train_raw)

    rankings = {}
    for model_name, model in models.items():
        X_bg = X_train_s if model_name == 'LogisticRegression' else X_train_raw
        X_ex = X_test_s if model_name == 'LogisticRegression' else X_test_raw

        shap_vals = compute_shap_values(model, model_name, X_bg, X_ex)
        ranking = get_feature_ranking(shap_vals, X_ex.columns)
        rankings[model_name] = ranking

    results[dataset_name] = rankings

    model_names = list(rankings.keys())
    for i in range(len(model_names)):
        for j in range(i+1, len(model_names)):
            m1, m2 = model_names[i], model_names[j]
            corr = spearman_stability(rankings[m1], rankings[m2])
            jac3 = jaccard_overlap(rankings[m1], rankings[m2], 3)
            jac5 = jaccard_overlap(rankings[m1], rankings[m2], 5)
            print(f"  {m1} vs {m2}: Spearman={corr:.3f}, Jaccard@3={jac3:.3f}, Jaccard@5={jac5:.3f}")

Processing German Credit...


  LogisticRegression vs RandomForest: Spearman=0.429, Jaccard@3=0.500, Jaccard@5=0.429
  LogisticRegression vs XGBoost: Spearman=0.421, Jaccard@3=0.500, Jaccard@5=0.429
  RandomForest vs XGBoost: Spearman=0.860, Jaccard@3=1.000, Jaccard@5=0.667
Processing UCI Credit Card...


  LogisticRegression vs RandomForest: Spearman=0.457, Jaccard@3=0.200, Jaccard@5=0.250
  LogisticRegression vs XGBoost: Spearman=0.269, Jaccard@3=0.200, Jaccard@5=0.111
  RandomForest vs XGBoost: Spearman=0.321, Jaccard@3=0.200, Jaccard@5=0.250
Processing GiveMeCredit...


  LogisticRegression vs RandomForest: Spearman=0.455, Jaccard@3=0.500, Jaccard@5=0.429
  LogisticRegression vs XGBoost: Spearman=0.503, Jaccard@3=0.500, Jaccard@5=0.429
  RandomForest vs XGBoost: Spearman=0.721, Jaccard@3=1.000, Jaccard@5=0.667
